In [1]:
pip install openml

In [2]:
pip install transformers datasets torch

In [3]:
import openml
import pandas as pd

In [4]:
data = openml.datasets.get_dataset(40975)
data

OpenML Dataset
Name.........: car
Version......: 3
Format.......: ARFF
Upload Date..: 2017-11-30 20:27:42
Licence......: Public
Download URL.: https://api.openml.org/data/v1/download/18116966/car.arff
OpenML URL...: https://www.openml.org/d/40975
# of features: None

In [5]:
X, y, categorical_indicator, attribute_names = data.get_data(target=data.default_target_attribute)

In [6]:
df = pd.DataFrame(X, columns=attribute_names)
df[data.default_target_attribute] = y
df.head()

,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


- buying: buying price
- maint: price of the maintenance
- doors: number of doors
- persons: capacity in terms of persons to carry
- lug_boot: the size of luggage boot
- safety: estimated safety of the car
- class: target

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1728 entries, 0 to 1727
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   buying    1728 non-null   category
 1   maint     1728 non-null   category
 2   doors     1728 non-null   category
 3   persons   1728 non-null   category
 4   lug_boot  1728 non-null   category
 5   safety    1728 non-null   category
 6   class     1728 non-null   category
dtypes: category(7)
memory usage: 13.1 KB


In [8]:
df['class'].value_counts()

,count
class,
unacc,1210
acc,384
good,69
vgood,65


- unacc — "unacceptable" (неприемлемый) Машина считается нежелательной для покупки (например, из-за плохой комбинации параметров: низкая безопасность, высокая цена и т. д.).

- acc — "acceptable" (приемлемый) Машина удовлетворяет минимальным требованиям, но не является отличным выбором.

- good — "good" (хороший) Машина хорошего качества с удачным сочетанием характеристик.

- vgood— "very good" (очень хороший) Машина с идеальными параметрами (например, низкая цена, высокая безопасность, вместительность и т. д.).

In [9]:
df['class'] = df['class'].replace({'unacc': 'unacceptable',
        'acc': 'acceptable',
        'good': 'good',
        'vgood': 'very good'})

<ipython-input-9-d8488c776396>:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df['class'] = df['class'].replace({'unacc': 'unacceptable',


In [10]:
df['class'].value_counts()

,count
class,
unacceptable,1210
acceptable,384
good,69
very good,65


In [11]:
df['safety'] = df['safety'].replace({'med': 'medium'})

<ipython-input-11-104e9516d0dc>:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df['safety'] = df['safety'].replace({'med': 'medium'})


In [12]:
df['maint'].value_counts()

,count
maint,
vhigh,432
high,432
med,432
low,432


In [13]:
df['maint'] = df['maint'].replace({'vhigh': 'very high', 'med': 'medium'})

<ipython-input-13-cbdb2b139785>:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df['maint'] = df['maint'].replace({'vhigh': 'very high', 'med': 'medium'})


In [14]:
df['lug_boot'] = df['lug_boot'].replace({'med': 'medium'})

<ipython-input-14-6782f7c6dfa7>:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df['lug_boot'] = df['lug_boot'].replace({'med': 'medium'})


In [15]:
df.sample(4)

,buying,maint,doors,persons,lug_boot,safety,class
1065,med,high,5more,4,medium,low,unacceptable
1415,low,high,2,4,small,high,acceptable
916,med,very high,3,more,big,medium,acceptable
575,high,high,3,2,big,high,unacceptable


In [16]:
def concatenate_car_features(row):
    buying_map = {
        'vhigh': 'very high',
        'high': 'high',
        'med': 'medium',
        'low': 'low'
    }

    text = (
        f"The car has a {buying_map[row['buying']]} buying price. "
        f"Maintenance costs are {row['maint']}. "
        f"It has {row['doors']} doors. "
        f"Passenger capacity is {row['persons']}. "
        f"Luggage boot size is {row['lug_boot']}. "
        f"Safety rating is {row['safety']}."
    )
    return text

In [17]:
# Применяем функцию к первой строке DataFrame
sample_text = concatenate_car_features(df.iloc[0])
print(sample_text)

The car has a very high buying price. Maintenance costs are very high. It has 2 doors. Passenger capacity is 2. Luggage boot size is small. Safety rating is low.


In [18]:
df['text'] = df.apply(lambda x: concatenate_car_features(x), axis=1)
df['text'].iloc[3]

'The car has a very high buying price. Maintenance costs are very high. It has 2 doors. Passenger capacity is 2. Luggage boot size is medium. Safety rating is low.'

In [19]:
from transformers import TrainingArguments, Trainer, AutoTokenizer
from transformers import AutoModelForSeq2SeqLM
from datasets import Dataset
import torch

In [20]:
dataset = Dataset.from_pandas(df[['text', 'class']])
dataset

Dataset({
    features: ['text', 'class'],
    num_rows: 1728
})

In [21]:
model = AutoModelForSeq2SeqLM.from_pretrained("bigScience/T0_3B")
tokenizer = AutoTokenizer.from_pretrained("bigScience/T0_3B")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/11.4G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.86k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [22]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 2048)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 2048)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=2048, out_features=2048, bias=False)
              (k): Linear(in_features=2048, out_features=2048, bias=False)
              (v): Linear(in_features=2048, out_features=2048, bias=False)
              (o): Linear(in_features=2048, out_features=2048, bias=False)
              (relative_attention_bias): Embedding(32, 32)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=2048, out_features=5120, bias=False)
              (wi_1): Linear(in_features=2048, out_features=5120, bias=False)
       

In [23]:
def tokenize_function(examples):
    inputs = tokenizer(examples['text'], padding="max_length", truncation=True, max_length=64, return_tensors="pt")
    labels = tokenizer(examples['class'], padding="max_length", truncation=True, max_length=16, return_tensors="pt")
    inputs['labels'] = labels['input_ids']
    return inputs

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/1728 [00:00<?, ? examples/s]

In [24]:
tokenized_dataset

Dataset({
    features: ['text', 'class', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 1728
})

In [25]:
split_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)

train_data = split_dataset['train']
test_data = split_dataset['test']

In [26]:
from collections import Counter

y_values = train_data['class']
if isinstance(y_values, list):
    print(Counter(y_values))

Counter({'unacceptable': 960, 'acceptable': 311, 'good': 59, 'very good': 52})


In [27]:
y_values_test = test_data['class']
if isinstance(y_values_test, list):
    print(Counter(y_values_test))

Counter({'unacceptable': 250, 'acceptable': 73, 'very good': 13, 'good': 10})


In [28]:
train_data = train_data.remove_columns(['text', 'class'])
test_data = test_data.remove_columns(['text', 'class'])

In [29]:
train_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

In [30]:
print("Форма input_ids:", train_data[0]['input_ids'].shape)
print("Форма attention_mask:", train_data[0]['attention_mask'].shape)
print("Форма labels:", train_data[0]['labels'].shape)

Форма input_ids: torch.Size([64])
Форма attention_mask: torch.Size([64])
Форма labels: torch.Size([16])


In [31]:
train_data['attention_mask'][0]

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [32]:
train_data['input_ids'][0]

tensor([   37,   443,    65,     3,     9,   306,  2611,   594,     5, 18192,
         1358,    33,  2768,     5,    94,    65,   314,  3377,     5,  3424,
           35,  1304,  2614,    19,  1682,   301, 13917,   545,  7378,   812,
           19,   600,     5,  6859,  5773,    19,   306,     5,     1,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0])

In [33]:
train_data['labels'][0]

tensor([29452,     1,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0])

In [34]:
for param in model.encoder.parameters():
    param.requires_grad = False

# Заморозка всех слоев в декодере, кроме 23-го блока
for i, layer in enumerate(model.decoder.block):
    if i != 23:  # Указание на 23-й слой (индекс 23)
        for param in layer.parameters():
            param.requires_grad = False

# Проверка заморозки
for name, param in model.named_parameters():
    print(f"{name}: requires_grad = {param.requires_grad}")

shared.weight: requires_grad = False
encoder.block.0.layer.0.SelfAttention.q.weight: requires_grad = False
encoder.block.0.layer.0.SelfAttention.k.weight: requires_grad = False
encoder.block.0.layer.0.SelfAttention.v.weight: requires_grad = False
encoder.block.0.layer.0.SelfAttention.o.weight: requires_grad = False
encoder.block.0.layer.0.SelfAttention.relative_attention_bias.weight: requires_grad = False
encoder.block.0.layer.0.layer_norm.weight: requires_grad = False
encoder.block.0.layer.1.DenseReluDense.wi_0.weight: requires_grad = False
encoder.block.0.layer.1.DenseReluDense.wi_1.weight: requires_grad = False
encoder.block.0.layer.1.DenseReluDense.wo.weight: requires_grad = False
encoder.block.0.layer.1.layer_norm.weight: requires_grad = False
encoder.block.1.layer.0.SelfAttention.q.weight: requires_grad = False
encoder.block.1.layer.0.SelfAttention.k.weight: requires_grad = False
encoder.block.1.layer.0.SelfAttention.v.weight: requires_grad = False
encoder.block.1.layer.0.SelfAtt

In [35]:
training_args = TrainingArguments(
    output_dir='./results',
    save_total_limit=2,
    run_name="my_t0_experiment",
    report_to="tensorboard",
    logging_strategy="steps",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    disable_tqdm=False,       # Показать прогресс-бар
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
)

In [36]:
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss,Validation Loss
50,1.923700,1.166227
100,0.074600,0.053601
150,0.047400,0.041026
200,0.052700,0.036570
250,0.043600,0.033341
300,0.047700,0.030719
350,0.034900,0.029669
400,0.040700,0.027555
450,0.032600,0.028369
500,0.034600,0.024348


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=870, training_loss=0.7199434008749052, metrics={'train_runtime': 2287.7539, 'train_samples_per_second': 6.041, 'train_steps_per_second': 0.38, 'total_flos': 1.477412568170496e+16, 'train_loss': 0.7199434008749052, 'epoch': 10.0})

In [37]:
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score

In [38]:
def decode_labels(labels):
    decoded_labels = []
    for label in labels:
        decoded_label = tokenizer.decode(label, skip_special_tokens=True).strip()
        decoded_labels.append(decoded_label)
    return decoded_labels

In [40]:
import numpy as np

In [52]:
def predict_class(input_ids):
    device = next(model.parameters()).device
    prompt = "How would you rate the decision to buy this car? Unacceptable, acceptable, good or very good? Answer only with the label, nothing else"

    # Токенизируем промпт
    prompt_ids = tokenizer(prompt, return_tensors="pt", max_length=64, truncation=True).to(device)

    # Объединяем input_ids и prompt_ids
    input_ids = input_ids.to(device)
    combined_input_ids = torch.cat([input_ids, prompt_ids['input_ids']], dim=-1)

    # Генерируем ответ модели с вероятностями
    with torch.no_grad():
        outputs = model.generate(
            input_ids=combined_input_ids,
            max_length=64,
            return_dict_in_generate=True,
            output_scores=True
        )

    predicted_text = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True).strip().lower()

    # Определяем класс
    predicted_class = next((c for c in ['unacceptable', 'acceptable', 'good', 'very good'] if c in predicted_text), None)

    # Получаем вероятности для первого токена ответа
    if len(outputs.scores) > 0:
        probs = torch.softmax(outputs.scores[0], dim=-1)
        class_probs = {
            'unacceptable': probs[0, tokenizer.encode('unacceptable')[0]].item(),
            'acceptable': probs[0, tokenizer.encode('acceptable')[0]].item(),
            'good': probs[0, tokenizer.encode('good')[0]].item(),
            'very good': probs[0, tokenizer.encode('very good')[0]].item()
        }
    else:
        raise ValueError("Model did not return any scores. Check model.generate() parameters")

    return predicted_class, predicted_text, class_probs

In [67]:
def get_predictions(test_data, sample_range=None):
    class_map = {'unacceptable': 0, 'acceptable': 1, 'good': 2, 'very good': 3}
    predictions = []
    true_labels = []
    predicted_texts = []
    all_probs = []

    decoded_true_labels = [tokenizer.decode(ids, skip_special_tokens=True).strip().lower()
                         for ids in test_data['labels']]

    for i in range(len(test_data)):
        example = test_data[i]
        true_class_text = decoded_true_labels[i]

        # Получаем предсказание и вероятности
        predicted_class, predicted_text, class_probs = predict_class(example['input_ids'].unsqueeze(0))

        # Проверяем валидность классов
        if (predicted_class is not None and
            true_class_text in ['unacceptable', 'acceptable', 'good', 'very good']):

            predictions.append(class_map[predicted_class])
            true_labels.append(class_map[true_class_text])
            predicted_texts.append(predicted_text)

            # Упорядочиваем вероятности
            ordered_probs = [
                class_probs['unacceptable'],
                class_probs['acceptable'],
                class_probs['good'],
                class_probs['very good']
            ]
            all_probs.append(ordered_probs)

        # Отладочный вывод
        if sample_range and sample_range[0] <= i < sample_range[1]:
            print(f"Example {i}:")
            print(f"  True: {true_class_text}")
            print(f"  Pred: {predicted_class or 'INVALID'}")
            print(f"  Probs: {class_probs}")
            print("-" * 40)

    return (np.array(true_labels),
            np.array(predictions),
            np.array(all_probs))

In [68]:
from sklearn.preprocessing import label_binarize

def calculate_metrics(true_labels, predictions, probs):
    # Проверяем и выравниваем массивы
    true_labels = np.array(true_labels).flatten()
    predictions = np.array(predictions).flatten()

    # Бинаризуем метки для ROC-AUC
    y_true_bin = label_binarize(true_labels, classes=[0, 1, 2, 3])

    # Вычисляем метрики
    metrics = {
        'accuracy': accuracy_score(true_labels, predictions),
        'f1': f1_score(true_labels, predictions, average='weighted'),
        'precision': precision_score(true_labels, predictions, average='weighted'),
        'recall': recall_score(true_labels, predictions, average='weighted'),
        'roc_auc': roc_auc_score(y_true_bin, class_probs, multi_class='ovo', average='weighted')
    }

    for name, value in metrics.items():
        print(f"{name}: {value:.4f}")

In [70]:
true_labels, predictions, class_probs = get_predictions(test_data)
calculate_metrics(true_labels, predictions, class_probs)

accuracy: 0.6647
f1: 0.6734
precision: 0.7615
recall: 0.6647
roc_auc: 0.8518


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [71]:
# Пример использования
start_idx = 0  # Начальный индекс выборки
end_idx = 149   # Конечный индекс выборки
true_labels_sample, predictions_sample, _ = get_predictions(test_data, sample_range=(start_idx, end_idx))

Example 0:
  True: acceptable
  Pred: unacceptable
  Probs: {'unacceptable': 0.4947562515735626, 'acceptable': 0.49178463220596313, 'good': 0.00837091263383627, 'very good': 0.005056306719779968}
----------------------------------------
Example 1:
  True: unacceptable
  Pred: unacceptable
  Probs: {'unacceptable': 0.9630512595176697, 'acceptable': 0.03655058518052101, 'good': 0.00015437875117640942, 'very good': 0.00022369752696249634}
----------------------------------------
Example 2:
  True: unacceptable
  Pred: unacceptable
  Probs: {'unacceptable': 0.7197167873382568, 'acceptable': 0.2591993808746338, 'good': 0.01764586567878723, 'very good': 0.003362600225955248}
----------------------------------------
Example 3:
  True: unacceptable
  Pred: unacceptable
  Probs: {'unacceptable': 0.9222854971885681, 'acceptable': 0.07764115929603577, 'good': 5.8384277508594096e-05, 'very good': 1.0954880053759553e-05}
----------------------------------------
Example 4:
  True: unacceptable
  Pre

In [72]:
from collections import Counter

# Ваш существующий код получения предсказаний
true_labels, predictions, probs = get_predictions(test_data)

# 1. Подсчет предсказанных классов
prediction_counts = Counter(predictions)
print("\nРаспределение предсказанных классов:")
print(prediction_counts)

# 2. Подсчет истинных классов (если нужно)
true_label_counts = Counter(true_labels)
print("\nРаспределение истинных классов:")
print(true_label_counts)

# 3. Красивое отображение с названиями классов
class_names = ['unacceptable', 'acceptable', 'good', 'very good']
print("\nДетальное распределение:")
for class_id, count in prediction_counts.items():
    print(f"{class_names[class_id]}: {count} примеров ({count/len(predictions):.1%})")


Распределение предсказанных классов:
Counter({np.int64(0): 188, np.int64(1): 158})

Распределение истинных классов:
Counter({np.int64(0): 250, np.int64(1): 73, np.int64(3): 13, np.int64(2): 10})

Детальное распределение:
acceptable: 158 примеров (45.7%)
unacceptable: 188 примеров (54.3%)


In [66]:
prediction_counts = Counter(predictions)

print(prediction_counts)

Counter({np.int64(1): 175, np.int64(0): 171})


In [73]:
model.save_pretrained("./saved_model")
tokenizer.save_pretrained("./saved_model")

('./saved_model/tokenizer_config.json',
 './saved_model/special_tokens_map.json',
 './saved_model/spiece.model',
 './saved_model/added_tokens.json',
 './saved_model/tokenizer.json')